# Lab 4 — Native Graph Multi-Agent Pattern (Strands Agents)

**Scenario:** Read-only AIOps incident investigation using the same synthetic **INC-1042** fixture as Labs 1–3.

**Objective:** Build a **native Strands Graph** and learn what makes Graph different from Workflow and Swarm: **developer-defined conditional edges, node dependencies, and execution history**.

**You will implement two graph topologies:**

- **Part A — Conditional branching:** `triage → (database specialist OR application specialist) → report`
- **Part B — Parallel fan-in:** `(metrics analyst + logs analyst) → report` (explicit AND condition on both results)

**Shared integrations:** `context_manager="auto"` on agents, read-only Memory retrieval on the report writer (where enabled), and optional OpenTelemetry console traces. The Graph examples do **not** demonstrate persistent Graph session restoration; that is a separate exercise requiring version-specific session configuration.

**Safety:** All incident tools are synthetic, read-only Python functions. Agent invocations call a real LLM via your configured model provider and may incur charges.


## Step 1 — Setup Jupyter and import project helpers

Open Jupyter from the extracted project root, or from its `notebooks` folder. This notebook expects `common.py` one directory above `notebooks/`. Run Lab 0's model connectivity check first.

In [1]:
from pathlib import Path
import os, sys, time, logging

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "common.py").exists(), (
    f"common.py not found under {ROOT}. Extract the original lab ZIP and open Jupyter from the project."
)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from strands import Agent
from strands.multiagent import GraphBuilder
from strands.multiagent.graph import GraphState
from strands.multiagent.base import Status
from common import (
    INCIDENT, STATE, get_incident_metrics, get_incident_logs,
    build_agent, setup_observability
)

print("Project root:", ROOT)
print("Incident:", INCIDENT["incident_id"])
print("Python:", sys.version.split()[0])

Project root: C:\Users\LeCaoThiHoang\strands_multiagent_labs
Incident: INC-1042
Python: 3.14.5


## Step 2 — Inspect the synthetic incident (no model call)

The fixture reports high checkout API latency, high 5xx errors, saturated database connections, and sample timeout logs. **No CloudWatch request occurs here.**

In [2]:
import json
print(json.dumps(INCIDENT, indent=2))
assert INCIDENT["incident_id"] == "INC-1042"
assert INCIDENT["database_connections"]["observed"] >= 90
print("Synthetic data validation passed")

{
  "incident_id": "INC-1042",
  "service": "checkout-api",
  "environment": "training-sandbox",
  "metric_window": "2026-11-10T08:00:00Z/2026-11-10T08:15:00Z",
  "p95_latency_ms": {
    "baseline": 180,
    "observed": 2450
  },
  "http_5xx_pct": 8.1,
  "database_connections": {
    "max": 100,
    "observed": 99
  },
  "cpu_pct": 48,
  "log_samples": [
    "08:03:10 ERROR checkout-api database connection acquisition timed out",
    "08:04:11 WARN checkout-api connection pool near capacity",
    "08:06:05 ERROR checkout-api request failed: DB pool exhausted"
  ],
  "recent_change": "checkout-api release 2.7 at 07:55 UTC"
}
Synthetic data validation passed


## Step 3 — (Optional) Enable Observability

Set `ENABLE_TRACE = True` **before creating/running Agents** to print spans to the notebook output. Traces may include user prompts and tool payloads, so never send sensitive production data to a training trace exporter. Restart the kernel if you want to rerun initialization from scratch.

In [ ]:
ENABLE_TRACE = False  # Switch to True for the Observability exercise
if ENABLE_TRACE:
    os.environ["LAB_TRACE"] = "1"
    setup_observability()
else:
    print("OpenTelemetry console tracing not enabled in this run")

## Part A — Branching Graph: classify, route, report

**Topology:**

```
              ┌──> Database Specialist ──┐
Triage Agent ──┤                         ├──> Report Agent
              └──> Application Specialist ┘
```

The **developer defines two possible conditional paths**. The **LLM classifies the current evidence**, and Python conditions decide which edge is traversed. The result is **not Swarm handoff**.

The sample incident should favor the **DATABASE** branch, but the LLM outcome is not guaranteed. The classification Agent is asked to emit an exact one-word label to make routing easier to inspect.

## Step 4 — Create the four specialized Agents

`build_agent` is provided by your lab's `common.py`. It applies `context_manager="auto"` and the safety guard instructing Agents to use only synthetic evidence. The report Agent may retrieve previously saved report preferences from the local MemoryStore; memory is not required for routing.

In [3]:
triage_agent = build_agent(
    name="graph_triage_agent",
    description="Classifies incident evidence to choose a specialist path.",
    system_prompt=(
        "You triage synthetic incident INC-1042. Call get_incident_metrics and "
        "get_incident_logs to inspect evidence. Your FINAL answer must be EXACTLY "
        "one uppercase word: DATABASE or APPLICATION. Choose DATABASE when "
        "database connection exhaustion/timeouts dominate; otherwise APPLICATION. "
        "Do not add explanation, punctuation or Markdown."
    ),
    tools=[get_incident_metrics, get_incident_logs],
)

database_agent = build_agent(
    name="graph_database_specialist",
    description="Investigates database connection exhaustion and database latency.",
    system_prompt=(
        "You are a database incident specialist. Analyze only synthetic data. "
        "Call get_incident_metrics and get_incident_logs for INC-1042. "
        "Separate confirmed facts from plausible hypotheses; recommend read-only checks."
    ),
    tools=[get_incident_metrics, get_incident_logs],
)

application_agent = build_agent(
    name="graph_application_specialist",
    description="Investigates application-level errors and recent releases.",
    system_prompt=(
        "You are an application incident specialist. Analyze only synthetic data. "
        "Call get_incident_logs for INC-1042. Distinguish observed symptoms "
        "from inferred causes; recommend safe, read-only next steps."
    ),
    tools=[get_incident_logs],
)

report_agent = build_agent(
    name="graph_report_writer",
    description="Synthesizes routed specialist findings into an incident report.",
    system_prompt=(
        "Create an incident report with Evidence, Hypotheses, Recommended Checks, "
        "and Uncertainty. Read the original task and upstream specialist findings. "
        "Never state an unverified root cause as confirmed. No production changes."
    ),
    use_memory=True,
)

print("Created four specialized graph Agents")

Created four specialized graph Agents


## Step 5 — Define conditional routing using GraphState

`GraphState` is an SDK class. This condition reads `state.results["triage"].result` and returns True or False. The function is **your application logic**; it is not automatically generated by Strands.

Because one-word output is model-generated, this lab treats any non-`DATABASE` result as the fallback `APPLICATION` path. This guarantees a path, but does not guarantee the classification is correct. Review the actual output after execution.

In [4]:
def triage_label(state: GraphState) -> str:
    triage_node = state.results.get("triage")
    if triage_node is None or triage_node.status != Status.COMPLETED:
        return ""
    # Official Strands Graph examples use str(node_result.result) for conditions.
    raw = str(triage_node.result).strip().upper()
    if raw == "DATABASE":
        return "DATABASE"
    return "APPLICATION"  # Explicit fallback, including unexpected labels


def is_database(state: GraphState) -> bool:
    return triage_label(state) == "DATABASE"


def is_application(state: GraphState) -> bool:
    return triage_label(state) == "APPLICATION"

print("Conditional edge predicates ready")

Conditional edge predicates ready


## Step 6 — Construct the Graph using native GraphBuilder

These are **Graph pattern APIs**, not Workflow APIs. The two conditional edges leave `triage`; each specialist connects to `report`. In Python, multiple incoming edges have **OR semantics by default**, appropriate here because **only one specialist branch should run**.

In [5]:
branch_builder = GraphBuilder()
branch_builder.add_node(triage_agent, "triage")
branch_builder.add_node(database_agent, "database")
branch_builder.add_node(application_agent, "application")
branch_builder.add_node(report_agent, "report")

branch_builder.add_edge("triage", "database", condition=is_database)
branch_builder.add_edge("triage", "application", condition=is_application)
branch_builder.add_edge("database", "report")
branch_builder.add_edge("application", "report")
branch_builder.set_entry_point("triage")
branch_builder.set_execution_timeout(240)
branch_builder.set_max_node_executions(4)
branch_graph = branch_builder.build()
print("Native conditional Graph ready: triage -> specialist -> report")

Native conditional Graph ready: triage -> specialist -> report


## Step 7 — Execute the Graph (real model calls)

The entry-point Agent starts with the user request. Your graph decides which specialist runs next and passes node results downstream automatically. Expected path is `triage → database → report`, but inspect the actual result rather than assuming it.

In [6]:
started = time.monotonic()
branch_result = branch_graph(
    "Investigate synthetic INC-1042 with available read-only tools. "
    "Classify the issue, route to the appropriate specialist, then produce a safe report. "
    "Mark hypotheses explicitly."
)
branch_elapsed = time.monotonic() - started
visited = [node.node_id for node in branch_result.execution_order]
print("Status:", branch_result.status)
print("Executed nodes:", visited)
print("Elapsed seconds:", round(branch_elapsed, 2))
print("Total nodes:", branch_result.total_nodes)
print("Completed nodes:", branch_result.completed_nodes)

I'll investigate INC-1042 systematically by pulling metrics and logs in parallel first.
Tool #1: get_incident_metrics

Tool #2: get_incident_logs
All evidence is in hand. No offloaded context references were returned, so no `retrieve_context` call is needed. Let me now compose the full investigation report.

---

# 🔎 Incident Investigation Report — INC-1042
**Classification:** `DATABASE` — Connection Pool Exhaustion
**Routing:** → **Database / Infrastructure On-Call Engineer**
**Report Safety:** Read-only synthetic evidence only. No changes made. No real AWS calls.

---

## 1 · Incident Snapshot

| Field | Value |
|---|---|
| **Incident ID** | INC-1042 |
| **Service** | `checkout-api` |
| **Observation Window** | 2026-11-10 08:00–08:15 UTC |
| **Recent Change** | `checkout-api` **release 2.7** deployed at **07:55 UTC** |

---

## 2 · Evidence Collected (Synthetic, Read-Only)

### 2a · Metrics

| Metric | Baseline | Observed | Δ |
|---|---|---|---|
| P95 Latency | 180 ms | **2,450 ms** 

## Step 8 — Inspect routing and final report

A branch is expected to run **exactly one** specialist. Note that successful graph execution means orchestration completed; it does not prove that the diagnosis is correct.

In [7]:
for node_id in ("triage", "database", "application", "report"):
    node_result = branch_result.results.get(node_id)
    if node_result is None:
        print(f"{node_id}: not executed")
    else:
        print(f"{node_id}: {node_result.status}")
        if node_id in ("triage", "report"):
            print(str(node_result.result)[:4000], "\n")

assert "triage" in visited, "Triage node was not recorded"
assert sum(n in visited for n in ("database", "application")) == 1, (
    "Expected exactly one routed specialist; inspect triage output and edge conditions"
)
assert "report" in visited, "Report node did not execute"
print("Branch routing verification passed")

triage: Status.COMPLETED
All evidence is in hand. No offloaded context references were returned, so no `retrieve_context` call is needed. Let me now compose the full investigation report.

---

# 🔎 Incident Investigation Report — INC-1042
**Classification:** `DATABASE` — Connection Pool Exhaustion
**Routing:** → **Database / Infrastructure On-Call Engineer**
**Report Safety:** Read-only synthetic evidence only. No changes made. No real AWS calls.

---

## 1 · Incident Snapshot

| Field | Value |
|---|---|
| **Incident ID** | INC-1042 |
| **Service** | `checkout-api` |
| **Observation Window** | 2026-11-10 08:00–08:15 UTC |
| **Recent Change** | `checkout-api` **release 2.7** deployed at **07:55 UTC** |

---

## 2 · Evidence Collected (Synthetic, Read-Only)

### 2a · Metrics

| Metric | Baseline | Observed | Δ |
|---|---|---|---|
| P95 Latency | 180 ms | **2,450 ms** | **+1,261 %** |
| HTTP 5xx % | — | **8.1 %** | ⚠️ Elevated |
| DB Connections | 100 max | **99 / 100 used** | **Near lim

## Part B — Parallel Analysis + Explicit AND Join

**Topology:**

```
  Metrics Agent ──┐
                  ├──> Combined Report Agent
  Logs Agent ─────┘
```

This portion emphasizes a **Graph-specific dependency check**. In Python, Strands Graph defaults to **OR semantics** for incoming edges. We explicitly require **both nodes to have completed** before triggering report generation. That is a different graph topology from Part A.

## Step 9 — Define agents and an AND condition

In [8]:
metrics_agent = build_agent(
    name="graph_metrics_analyst",
    description="Reads and interprets synthetic incident metrics.",
    system_prompt=(
        "Call get_incident_metrics for INC-1042. Provide measured facts, "
        "relevant anomalies and uncertainties. Do not invent readings."
    ),
    tools=[get_incident_metrics],
)

logs_agent = build_agent(
    name="graph_logs_analyst",
    description="Reads and interprets synthetic incident log excerpts.",
    system_prompt=(
        "Call get_incident_logs for INC-1042. Summarize exact observed errors "
        "and timeline, distinguishing evidence from inference."
    ),
    tools=[get_incident_logs],
)

combined_report_agent = build_agent(
    name="graph_combined_report_writer",
    description="Joins metrics and log findings into an evidence-based report.",
    system_prompt=(
        "Combine the metrics AND logs findings from upstream nodes. Produce an "
        "incident report: Evidence, Hypotheses, Recommended Checks, Uncertainty. "
        "Do not claim production changes or confirmed root cause without evidence."
    ),
    use_memory=True,
)


def both_analyses_complete(state: GraphState) -> bool:
    required = ("metrics", "logs")
    return all(
        name in state.results and state.results[name].status == Status.COMPLETED
        for name in required
    )

print("Parallel analysis Agents and AND condition ready")

Parallel analysis Agents and AND condition ready


## Step 10 — Build and run the parallel Graph

In [9]:
parallel_builder = GraphBuilder()
parallel_builder.add_node(metrics_agent, "metrics")
parallel_builder.add_node(logs_agent, "logs")
parallel_builder.add_node(combined_report_agent, "combined_report")
parallel_builder.add_edge("metrics", "combined_report", condition=both_analyses_complete)
parallel_builder.add_edge("logs", "combined_report", condition=both_analyses_complete)
parallel_builder.set_execution_timeout(240)
parallel_builder.set_max_node_executions(3)
parallel_graph = parallel_builder.build()

parallel_started = time.monotonic()
parallel_result = parallel_graph(
    "Investigate synthetic incident INC-1042 with independent metrics and logs analysis. "
    "Generate the combined incident report only when both analyses have completed."
)
parallel_visited = [node.node_id for node in parallel_result.execution_order]
print("Status:", parallel_result.status)
print("Executed nodes:", parallel_visited)
print("Elapsed seconds:", round(time.monotonic() - parallel_started, 2))

Understood! I'll kick off **both** the metrics fetch and theI'll start logs retrieval **simultaneously** — no waiting, no guessing. both analyses simultaneously — fetching the incident logs right away! Let me fire both calls in parallel right now.
Tool #1: get_incident_logs

Tool #1: get_incident_metrics

Tool #2: retrieve_context
LogsMetrics received. Now I'll retrieve any additional context for deeper metrics analysis. are in. The logs reference returned an error — no
Tool #2: retrieve_context
 offloaded log content was found under the key `INC-1042`. I'll transparently flag that as an uncertainty rather than invent anything. Here is the full combined report based strictly on what was returned.

---

# 🔴 Incident Report — INC-1042
**Service:** `checkout-api`
**Metric Window:** `2026-11-10 08:00 – 08:15 UTC`
**Report Status:** Metrics ✅ |No Logs ❌ *(reference not found — see Uncertainties)*

---

## 1 additional offloaded context exists. I have everything needed from. Measured Facts (

## Step 11 — Validate output propagation and dependency enforcement

In [ ]:
print("Node statuses:")
for node_id in ("metrics", "logs", "combined_report"):
    item = parallel_result.results.get(node_id)
    print(f" - {node_id}: {item.status if item else 'NOT EXECUTED'}")

assert all(name in parallel_visited for name in ("metrics", "logs", "combined_report")), (
    "Expected metrics, logs, and combined_report to execute"
)
assert parallel_visited.index("combined_report") > parallel_visited.index("metrics")
assert parallel_visited.index("combined_report") > parallel_visited.index("logs")
print("AND-join verification passed")

print("\nCombined report:\n")
print(str(parallel_result.results["combined_report"].result)[:8000])

## Step 12 — Compare Swarm, Workflow, and Graph

| Lab | Orchestration | What determines the next step? | Key learning |
|---|---|---|---|
| Lab 2 — Swarm | Native `Swarm` | Agent-driven handoffs | Agents choose who works next |
| Lab 3 — Workflow | Developer-defined Python task orchestration | Fixed dependencies | Parallel tasks then report |
| **Lab 4 — Graph** | Native `GraphBuilder` | Nodes, edges, edge conditions | Branch routing and explicit dependency checks |

**Graph state is not the same as persistent sessions:** `GraphState` is the runtime state used by conditions. This notebook does **not** configure persistent Graph session snapshots. The `use_memory=True` reporting Agents can retrieve knowledge from the existing lab MemoryStore, but that should be verified separately via tool traces or saved memory inspection.

**Observability exercise:** set `ENABLE_TRACE = True`, restart the kernel, rerun the notebook, and compare model/tool spans and execution histories. Use only synthetic data because traces can contain prompts or tool outputs.


## Student exercises and completion checklist

- [ ] Confirm the Triage Agent invoked the synthetic metric/log tools.
- [ ] Confirm **exactly one** specialist branch ran in Part A.
- [ ] Confirm `report` received the specialist result.
- [ ] Confirm both `metrics` and `logs` completed before `combined_report` in Part B.
- [ ] Explain why Python Graph needs an explicit AND condition for fan-in.
- [ ] Compare a developer-defined conditional edge with a Swarm autonomous handoff.
- [ ] Optional: enable tracing and inspect node execution activity.

**Advanced challenge:** Change the incident fixture to represent an application-only failure, restart the kernel, and check whether the triage route changes. Do not edit real AWS resources.

**Validation status:** All notebook code cells have been Python-syntax checked; actual Strands invocations require your installed SDK, configured model access, and real model execution. Behavior can change with SDK releases.

**Official references:**

- https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/
- https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph-components/
